# Apache Hudi Core Conceptions (2) - File Layouts

## 1. Configuration

In [1]:
%%sh
# deploy hudi bundle jar
hdfs dfs -copyFromLocal -f /usr/lib/hudi/hudi-spark-bundle.jar /tmp/hudi-spark-bundle.jar
hdfs dfs -ls /tmp/hudi-spark-bundle.jar

-rw-r--r--   1 emr-notebook hdfsadmingroup   61421977 2023-03-08 05:35 /tmp/hudi-spark-bundle.jar


In [2]:
%%configure -f
{
    "conf" : {
        "spark.jars":"hdfs:///tmp/hudi-spark-bundle.jar",            
        "spark.serializer":"org.apache.spark.serializer.KryoSerializer",
        "spark.sql.extensions":"org.apache.spark.sql.hudi.HoodieSparkSessionExtension",
        "spark.sql.catalog.spark_catalog":"org.apache.spark.sql.hudi.catalog.HoodieCatalog"
    }
}

ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
33,application_1678096020253_0053,spark,idle,Link,Link,None,


In [3]:
%env S3_BUCKET=apache-hudi-core-conceptions

env: S3_BUCKET=apache-hudi-core-conceptions


In [4]:
%%sql
set S3_BUCKET=apache-hudi-core-conceptions

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
35,application_1678096020253_0055,spark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [5]:
%env WORKSPACE=/home/emr-notebook/apache-hudi-core-conceptions

env: WORKSPACE=/home/emr-notebook/apache-hudi-core-conceptions


In [6]:
%%sh
# make workspace
mkdir -p $WORKSPACE
# deploy hudi-stat.sh, a utility shell script to output hudi table status
wget https://github.com/bluishglc/hudi-core-conceptions/releases/download/v1.0/hudi-stat.sh -O $WORKSPACE/hudi-stat.sh &>/dev/null
chmod a+x $WORKSPACE/hudi-stat.sh
ls $WORKSPACE/hudi-stat.sh

/home/emr-notebook/apache-hudi-core-conceptions/hudi-stat.sh


In [7]:
%%html
<style>
table {float:left}
</style>

## 2. Test Case 1 - COW File Layouts

### 2.1. Test Plan

Step No.|Action|Volume Per Partition |Storage
:--------:|:------|:------|:----------
1|Insert|96MB|+1 Small File
2|Insert|14.6MB|+1 Big File
3|Insert|3.7MB|+1 Small File
4|Insert|188.5MB|+1 Big File, +1 Small File

### 2.2. Key Settings

KEY|DEFAULT VALUE|SET VALUE
:---|:---|:---
hoodie.parquet.max.file.size|125829120 ( 120MB )|Default Value
hoodie.parquet.small.file.limit|104857600 ( 100MB )|Default Value
hoodie.copyonwrite.record.size.estimate|1024|175

### 2.3. Set Variables

In [8]:
%%sql
set TABLE_NAME=reviews_cow

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [9]:
%env TABLE_NAME=reviews_cow

env: TABLE_NAME=reviews_cow


In [10]:
%%sql
set TABLE_PATH=s3://${S3_BUCKET}/reviews_cow

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [11]:
%env TABLE_PATH=s3://${S3_BUCKET}/reviews_cow

env: TABLE_PATH=s3://${S3_BUCKET}/reviews_cow


### 2.4. Create Table

In [12]:
%%sh
aws s3 rm ${TABLE_PATH} --recursive &>/dev/null
rm -rf ~/${TABLE_NAME}
sleep 5

reviews_cow


In [13]:
%%sql
drop table if exists ${TABLE_NAME}

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [14]:
%%sql
create table if not exists ${TABLE_NAME}(
    review_id string, 
    star_rating int, 
    review_body string, 
    review_date date, 
    year long,
    timestamp long,
    parity int
)
using hudi
location '${TABLE_PATH}'
partitioned by (parity)
options ( 
    type = 'cow',  
    primaryKey = 'review_id', 
    preCombineField = 'timestamp',
    hoodie.copyonwrite.record.size.estimate = '175'
);

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

### 2.5. Insert 96MB ( +1 Small File )

In [15]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 2003;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [16]:
%%sh
${WORKSPACE}/hudi-stat.sh ${TABLE_PATH} timeline commits storage


[ TIMELINE ]


[ COMMITS ]


[ STORAGE ]

/home/emr-notebook/reviews_cow
    6 used in 0 directories, 0 files


### 2.6. Insert 14.6MB ( +1 Big File )

In [17]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 1998;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [18]:
%%sh
${WORKSPACE}/hudi-stat.sh ${TABLE_PATH} timeline commits storage


[ TIMELINE ]


[ COMMITS ]


[ STORAGE ]

/home/emr-notebook/reviews_cow
    6 used in 0 directories, 0 files


### 2.7. Insert 3.7MB ( +1 Small File )

In [19]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 1997;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [20]:
%%sh
${WORKSPACE}/hudi-stat.sh ${TABLE_PATH} timeline commits storage


[ TIMELINE ]


[ COMMITS ]


[ STORAGE ]

/home/emr-notebook/reviews_cow
    6 used in 0 directories, 0 files


### 2.8. Insert 188.5MB ( +1 Big File, +1 Small File )

In [21]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 2007;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [22]:
%%sh
${WORKSPACE}/hudi-stat.sh ${TABLE_PATH} timeline commits storage


[ TIMELINE ]


[ COMMITS ]


[ STORAGE ]

/home/emr-notebook/reviews_cow
    6 used in 0 directories, 0 files


## 3. Test Case 2 -  MOR File Layouts

## 4. Test Case 3 -  Default Settings

### 4.1. Set Variables

In [23]:
%%sql
set TABLE_NAME=reviews_cow_simple_3

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [24]:
%env TABLE_NAME=reviews_cow_simple_3

env: TABLE_NAME=reviews_cow_simple_3


In [25]:
%%sql
set TABLE_PATH=s3://${S3_BUCKET}/reviews_cow_simple_3

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [26]:
%env TABLE_PATH=s3://${S3_BUCKET}/reviews_cow_simple_3

env: TABLE_PATH=s3://${S3_BUCKET}/reviews_cow_simple_3


## 4.2 Create Table

In [27]:
%%sh
echo $(basename ${TABLE_PATH})
aws s3 rm ${TABLE_PATH} --recursive &>/dev/null
rm -rf ~/${TABLE_NAME}
sleep 5

reviews_cow_simple_3


In [28]:
%%sql
drop table if exists ${TABLE_NAME}

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [29]:
%%sql
create table if not exists ${TABLE_NAME}(
    review_id string, 
    star_rating int, 
    review_body string, 
    review_date date, 
    year long,
    timestamp long,
    parity int
)
using hudi
location '${TABLE_PATH}'
partitioned by (parity)
options ( 
    type = 'cow',  
    primaryKey = 'review_id', 
    preCombineField = 'timestamp'
    -- hoodie.index.type = 'SIMPLE',
    -- hoodie.copyonwrite.record.size.estimate = '175'
);

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

### 4.3. Batch 1 - Insert ( 0 -> 96MB / Partition )

In [30]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 2003;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [31]:
%%sh
${WORKSPACE}/hudi-stat.sh ${TABLE_PATH} timeline commits storage


[ TIMELINE ]


[ COMMITS ]


[ STORAGE ]

/home/emr-notebook/reviews_cow_simple_3
    6 used in 0 directories, 0 files


### 4.3. Batch 2 - Insert ( 96MB -> 417MB / Partition )

In [32]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 2010;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [33]:
%%sh
${WORKSPACE}/hudi-stat.sh ${TABLE_PATH} timeline commits storage


[ TIMELINE ]


[ COMMITS ]


[ STORAGE ]

/home/emr-notebook/reviews_cow_simple_3
    6 used in 0 directories, 0 files
